# Notebook 20 — why not just ask the model how sure it is?

The first question a reviewer asks of any sampling-based uncertainty method.
Entropy needs K forward passes; verbalized confidence needs one. If simply
asking works as well, the expensive method is hard to justify.

This project has the answer for the **grading** arm only (notebook 04: AUROC
**0.547 [0.432, 0.661]**, no signal). The **perception** arm — the one that
actually works, and the one the paper leads with — has never been tested.

Same 300 images, same seed, same K=5, same model (Qwen2.5-VL-3B). The prompt
adds a third field:

```
**Confidence:**<an integer from 0 to 100>
```

**Design choice worth stating:** the confidence is requested *after* the
transcription, so the model commits to a reading before rating it. Asking
first invites the rating to drive the answer, which measures something else.

Because confidence is collected on every one of the K samples, this run gives
a **paired** comparison on identical items:

| score | source |
|---|---|
| perception entropy | disagreement across the K samples |
| −mean verbalized confidence | the model's own self-report |

The headline is the **paired bootstrap difference**, not two separate AUROCs
— comparing overlapping marginal CIs is the difference-in-significance
fallacy this project has already been burned by.

**What each outcome means.** If entropy wins by a resolved margin, the
sampling cost is justified and the paper has its answer. If verbalized
confidence matches it, that is a genuinely important negative for the method
and must be reported: it would mean one forward pass buys what five do.

In [1]:
# Install. Qwen2.5-VL needs a current transformers; nothing exotic.
%pip install -q transformers accelerate datasets huggingface_hub bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 MB 53.7 MB/s eta 0:00:00:00:0100:01


In [2]:
# Auth & code access. Identical to notebooks 12-15, including the
# sys.modules purge that makes a re-clone actually take effect.
import json
import os
from getpass import getpass

from huggingface_hub import login

from google.colab import drive

drive.mount("/content/drive")
PROJECT_DIR = "/content/drive/MyDrive/uncertainty-math-vlm"
DRIVE_MODEL_CACHE = f"{PROJECT_DIR}/model_cache"
os.makedirs(DRIVE_MODEL_CACHE, exist_ok=True)

TOKEN_FILE = f"{PROJECT_DIR}/.tokens.json"
RESET_TOKENS = False


def get_token(name, prompt):
    tokens = {}
    if os.path.exists(TOKEN_FILE):
        with open(TOKEN_FILE) as f:
            tokens = json.load(f)
    if RESET_TOKENS or not tokens.get(name):
        tokens[name] = getpass(prompt).strip()
        with open(TOKEN_FILE, "w") as f:
            json.dump(tokens, f)
        os.chmod(TOKEN_FILE, 0o600)
    return tokens[name]


HF_TOKEN = get_token("HF_TOKEN", "Hugging Face token (asked once): ")
GH_TOKEN = get_token("GH_TOKEN", "GitHub token with 'repo' scope (asked once): ")
if not HF_TOKEN.startswith("hf_"):
    raise ValueError("Stored HF token does not start with 'hf_'; set RESET_TOKENS = True.")
login(token=HF_TOKEN)
print("Hugging Face login OK")

REPO_URL = "https://github.com/sepehrmaleki369/uncertainty-math-vlm.git"
!rm -rf repo
!git clone -q {REPO_URL} repo
%pip install -q -e repo/

import importlib
import sys

REPO_DIR = os.path.abspath("repo")
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
importlib.invalidate_caches()
for _n in [m for m in sys.modules if m == "pilot" or m.startswith("pilot.")]:
    del sys.modules[_n]

import pilot.data
import pilot.prompts
import pilot.parsing
import pilot.entropy
import pilot.canonicalize
import pilot.plotting

print("pilot imported from:", os.path.dirname(pilot.__file__))

Mounted at /content/drive
Hugging Face login OK
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 144.2/144.2 kB 4.6 MB/s eta 0:00:00
  Building editable for pilot (pyproject.toml) ... done
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
omegaconf 2.3.1 requires antlr4-python3-runtime==4.9.*, but you have antlr4-python3-runtime 4.11.0 which is incompatible.
pilot imported from: /content/repo/pilot


In [3]:
# Model load. Qwen2.5-VL-3B -- the SAME model as the reference n=300 run.
# Changing the model and the prompt at once would make this uninterpretable.
import torch
from transformers import AutoProcessor, Qwen2_5_VLForConditionalGeneration

# SECONDARY EXPERIMENT, one line: swap to Qwen2.5-VL-7B-Instruct.
#
# 7B has only ever been run on the GRADING arm in this project -- perception
# at 7B is untested, so this also answers "does the perception signal hold as
# a family scales?". Lower priority than the 3B run: Pixtral-12B already
# provides the second model FAMILY, and this run's purpose is the extractor
# manipulation, which needs the 3B baseline to compare against. Do 3B first.
#
# The checkpoint path below is keyed by the model slug, so switching models
# starts a fresh checkpoint instead of silently resuming into another model's
# samples.
MODEL_ID = "Qwen/Qwen2.5-VL-3B-Instruct"
# MODEL_ID = "Qwen/Qwen2.5-VL-7B-Instruct"   # <-- the 7B variant
QUANTIZED = False          # bf16, matching the reference run

processor = AutoProcessor.from_pretrained(MODEL_ID)
model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    MODEL_ID, torch_dtype=torch.bfloat16, device_map="auto")
model.eval()
print(f"loaded {MODEL_ID}  quantized={QUANTIZED}  dtype={model.dtype}")

preprocessor_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

chat_template.json:   0%|          | 0.00/1.05k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.37k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/5.70k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/824 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/216 [00:00<?, ?B/s]

loaded Qwen/Qwen2.5-VL-3B-Instruct  quantized=False  dtype=torch.bfloat16


In [4]:
# Sample. ALWAYS drawn at n=300 -- the same call every reference run used, so
# the comparison is apples-to-apples -- then truncated to PROCESS_N for the
# gate. Because the draw is identical, the gate items are a prefix of the full
# run and the checkpoint carries straight over. Set PROCESS_N = 300 and re-run
# the generation cell to continue; the first 50 are not regenerated.
import logging

import pilot.data

logging.basicConfig(level=logging.INFO)

SAMPLE_N = 300
SEED = 42
TARGET_ERROR_FRAC = 0.5
PROCESS_N = 300          # <-- the gate. Raise to 300 only after the gate passes.

full_sample = pilot.data.load_fermat_balanced(
    n=SAMPLE_N, seed=SEED, target_error_frac=TARGET_ERROR_FRAC)
sample = full_sample.select(range(PROCESS_N))
print(f"drawn {len(full_sample)} items, processing the first {len(sample)}")
print(f"has_error in this slice: {sum(bool(x) for x in sample['has_error'])}/{len(sample)}")

README.md:   0%|          | 0.00/3.74k [00:00<?, ?B/s]

data/train-00000-of-00010.parquet: reconstructing file:   0%|          |  0.00B /  467MB            

data/train-00000-of-00010.parquet: downloading bytes:           |  0.00B            

data/train-00001-of-00010.parquet: reconstructing file:   0%|          |  0.00B /  481MB            

data/train-00001-of-00010.parquet: downloading bytes:           |  0.00B            

...

data/train-00008-of-00010.parquet: downloading bytes:           |  0.00B            

data/train-00009-of-00010.parquet: reconstructing file:   0%|          |  0.00B /  493MB            

data/train-00009-of-00010.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/2244 [00:00<?, ? examples/s]

drawn 300 items, processing the first 300
has_error in this slice: 150/300


In [5]:
# Adapter + pre-flight. Same shape as notebook 19; only the prompt differs.
import pilot.prompts


def qwen_inputs(messages):
    text = processor.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True)
    images = [c["image"] for m in messages for c in m["content"]
              if c.get("type") == "image"]
    return processor(text=[text], images=images, return_tensors="pt").to(model.device)


# Pre-flight: does the model actually emit a parseable **Confidence:** field?
# If it does not, the run measures format compliance rather than confidence.
import pilot.parsing

_probe = qwen_inputs(pilot.prompts.build_transcription_messages_confidence(
    full_sample[0]["image"]))
with torch.no_grad():
    _out = model.generate(**_probe, max_new_tokens=320, do_sample=False)
_text = processor.batch_decode(_out[:, _probe["input_ids"].shape[1]:],
                               skip_special_tokens=True)[0]
print(_text[:800])
print("\n" + "=" * 70)
print("parsed confidence:", pilot.parsing.parse_confidence(_text))
print("If this is None on a few probes, stop -- the gate will fail anyway.")

**Question:**
Find the number of 4-letter words, with or without meaning, which can be formed out of the letters of the word ROSE, where the repetition of the letters is not allowed.

**Answer:**
There are many ways of filling in 4 vacant places by the 4 letters, keeping in mind that the repetition is not allowed. The first place can be filled in 4 different ways by anyone of the 4 letters R, O, S, E. Following which, the second place can be filled in by anyone of the remaining 3 letters in 3 different ways, following which the third place can be filled in 2 different ways; following which, the fourth place can be filled in 1 way. Additionally, if we consider the factorial function, n!, which denotes the product of all positive integers up to n, we have 4! = 4 x 3 x 2 x 1 = 24. Furthermore

parsed confidence: 95
If this is None on a few probes, stop -- the gate will fail anyway.


In [6]:
# Generation: K=5, transcription + a verbalized confidence per sample.
# Batch-backoff ladder and per-item Drive checkpoint, both carried over from
# notebook 16 unchanged. The checkpoint is keyed by SAMPLE_N (300), NOT by
# PROCESS_N, so raising PROCESS_N resumes rather than restarts.
import gc
import json
import os
import time

from tqdm.auto import tqdm

K_TRANSCRIPTION = 5
TEMP = 0.7
_LADDER = [5, 2, 1]
_state = {"i": 0}
META = ("orig_q", "pert_a", "has_error", "handwriting_style", "image_quality")
INFRA = (ConnectionError, TimeoutError, torch.cuda.OutOfMemoryError, OSError)


def _batch(messages, n, temperature):
    inputs = qwen_inputs(messages)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=512, do_sample=True,
                             temperature=temperature, num_return_sequences=n)
    texts = processor.batch_decode(out[:, inputs["input_ids"].shape[1]:],
                                   skip_special_tokens=True,
                                   clean_up_tokenization_spaces=False)
    del out, inputs
    gc.collect(); torch.cuda.empty_cache()
    return texts


def generate_k(messages, n, temperature):
    texts, last = [], None
    while len(texts) < n:
        size = min(_LADDER[_state["i"]], n - len(texts))
        for attempt in range(3):
            try:
                texts += _batch(messages, size, temperature)
                last = None
                break
            except torch.cuda.OutOfMemoryError:
                gc.collect(); torch.cuda.empty_cache()
                if _state["i"] + 1 < len(_LADDER):
                    _state["i"] += 1
                    print(f"  OOM at batch {size}; dropping to {_LADDER[_state['i']]}",
                          flush=True)
                    size = min(_LADDER[_state["i"]], n - len(texts))
                    continue
                raise
            except INFRA as exc:
                last = exc
                gc.collect(); torch.cuda.empty_cache()
                if attempt < 2:
                    time.sleep(5)
        if last is not None:
            raise last
    return texts


CHECKPOINT_DIR = f"{PROJECT_DIR}/checkpoints"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
slug = MODEL_ID.split("/")[-1]
ckpt = (f"{CHECKPOINT_DIR}/conf_perc_{slug}_n{SAMPLE_N}_seed{SEED}"
        f"_k{K_TRANSCRIPTION}{'_4bit' if QUANTIZED else ''}.jsonl")

raw_results = []
if os.path.exists(ckpt):
    with open(ckpt) as f:
        raw_results = [json.loads(l) for l in f if l.strip()]
    valid = []
    for idx, e in enumerate(raw_results):
        if idx >= len(full_sample):
            break
        it = full_sample[idx]
        if not all(e["item"].get(k) == it[k] for k in META):
            print(f"checkpoint item {idx+1} mismatch; resuming there."); break
        if len(e.get("transcription_samples_raw", [])) != K_TRANSCRIPTION:
            break
        valid.append(e)
    if len(valid) != len(raw_results):
        with open(ckpt, "w") as f:
            for e in valid:
                f.write(json.dumps(e, default=str) + "\n")
    raw_results = valid
    print(f"resuming from {len(raw_results)} completed items")

todo = [i for i in range(PROCESS_N) if i >= len(raw_results)]
if not todo:
    print(f"first {PROCESS_N} items already done.")
else:
    print(f"generating items {todo[0]+1}..{todo[-1]+1}", flush=True)
    with tqdm(total=len(todo) * K_TRANSCRIPTION, desc="confidence", unit="sample") as pbar:
        for idx in todo:
            item = full_sample[idx]
            t0 = time.time()
            tr = generate_k(
                pilot.prompts.build_transcription_messages_confidence(item["image"]),
                K_TRANSCRIPTION, TEMP)
            pbar.update(K_TRANSCRIPTION)
            entry = {"item": {k: item[k] for k in META},
                     "transcription_samples_raw": tr,
                     "quantized": QUANTIZED, "elapsed_seconds": time.time() - t0}
            raw_results.append(entry)
            with open(ckpt, "a") as f:
                f.write(json.dumps(entry, default=str) + "\n"); f.flush()
            print(f"  item {idx+1}/{PROCESS_N}: {time.time()-t0:.1f}s", flush=True)

print(f"raw_results: {len(raw_results)} items")

generating items 1..300


confidence:   0%|          | 0/1500 [00:00<?, ?sample/s]

  item 1/300: 18.8s
  item 2/300: 13.2s
  item 3/300: 10.9s
  item 4/300: 16.3s
  item 5/300: 22.6s
  item 6/300: 12.9s
  item 7/300: 12.2s
  item 8/300: 15.0s
  item 9/300: 14.5s
  item 10/300: 18.5s
  item 11/300: 9.3s
  item 12/300: 19.4s
  item 13/300: 21.6s
  item 14/300: 15.9s
  item 15/300: 11.4s
  item 16/300: 11.3s
  item 17/300: 12.4s
  item 18/300: 14.7s
  item 19/300: 11.7s
  item 20/300: 24.6s
  item 21/300: 25.2s
  item 22/300: 13.1s
  item 23/300: 12.6s
  item 24/300: 21.7s
  item 25/300: 11.9s
  item 26/300: 16.6s
  item 27/300: 24.4s
  item 28/300: 11.0s
  item 29/300: 24.5s
  item 30/300: 6.7s
  item 31/300: 16.3s
  item 32/300: 9.3s
  item 33/300: 16.6s
  item 34/300: 7.2s
  item 35/300: 15.1s
  item 36/300: 17.6s
  item 37/300: 20.7s
  item 38/300: 16.4s
  item 39/300: 11.3s
  item 40/300: 19.4s
  item 41/300: 14.5s
  item 42/300: 16.0s
  item 43/300: 20.0s
  item 44/300: 7.5s
  item 45/300: 14.9s
  item 46/300: 13.7s
  item 47/300: 16.8s
  item 48/300: 9.3s
  item 

In [7]:
# THE COMPARISON. Entropy vs the model's own self-report, on identical items.
import math

import pandas as pd

import pilot.parsing
import pilot.rescore
from pilot.plotting import bootstrap_auroc_ci, bootstrap_auroc_difference_ci

rows = []
for e in raw_results:
    tr = e["transcription_samples_raw"]
    scored = pilot.rescore.score_item(tr, e["item"]["pert_a"], "strict_v1")
    confs = [pilot.parsing.parse_confidence(s) for s in tr]
    got = [c for c in confs if c is not None]
    rows.append({**{k: e["item"][k] for k in META},
                 "perception_entropy": scored["perception_entropy"],
                 "transcription_correct": scored["transcription_correct"],
                 "n_transcription_parse_failures":
                     scored["n_transcription_parse_failures"],
                 "n_confidence_parsed": len(got),
                 "mean_confidence": sum(got) / len(got) if got else float("nan"),
                 "all_transcription_samples_raw": tr,
                 "quantized": e["quantized"], "model_id": MODEL_ID,
                 "k_transcription": K_TRANSCRIPTION})
scored_df = pd.DataFrame(rows)

# Higher confidence should mean LESS likely wrong, so negate before feeding it
# to an AUROC that predicts "is wrong" -- the same convention as the token
# logprob baselines. Getting this backwards silently inverts the result.
scored_df["neg_confidence"] = -scored_df["mean_confidence"]

parse_rate = scored_df["n_confidence_parsed"].sum() / (len(scored_df) * K_TRANSCRIPTION)
usable = scored_df.dropna(subset=["mean_confidence"])
print(f"n = {len(scored_df)}   confidence parsed on {parse_rate:.1%} of samples, "
      f"{len(usable)}/{len(scored_df)} items usable")
print(f"stated confidence: min {usable.mean_confidence.min():.0f}  "
      f"median {usable.mean_confidence.median():.0f}  "
      f"max {usable.mean_confidence.max():.0f}  "
      f"distinct values {usable.mean_confidence.nunique()}")

if parse_rate < 0.80:
    print("\nGATED: fewer than 80% of samples produced a parseable confidence. "
          "This run measures format compliance, not confidence.")
elif len(scored_df) < 300:
    print(f"\nPROCESS_N={len(scored_df)} < 300 -- no AUROC printed by design.")
else:
    ent = bootstrap_auroc_ci(usable, "perception_entropy",
                             "transcription_correct", n_boot=10000, seed=0)
    con = bootstrap_auroc_ci(usable, "neg_confidence",
                             "transcription_correct", n_boot=10000, seed=0)
    print(f"\n  perception entropy      AUROC {ent['auroc']:.3f} "
          f"[{ent['ci_low']:.3f}, {ent['ci_high']:.3f}]")
    print(f"  -verbalized confidence  AUROC {con['auroc']:.3f} "
          f"[{con['ci_low']:.3f}, {con['ci_high']:.3f}]")

    # The headline. Paired on the same resampled items -- comparing the two
    # marginal CIs above would be the difference-in-significance fallacy.
    d = bootstrap_auroc_difference_ci(
        usable, "perception_entropy", "transcription_correct",
        "neg_confidence", "transcription_correct", n_boot=10000, seed=0)
    resolved = d["ci_low"] > 0 or d["ci_high"] < 0
    print(f"\n  PAIRED entropy - confidence: {d['difference']:+.3f} "
          f"[{d['ci_low']:+.3f}, {d['ci_high']:+.3f}]  resolved={resolved}")
    if resolved and d["difference"] > 0:
        print("  -> entropy beats simply asking. The K forward passes are justified.")
    elif resolved:
        print("  -> ASKING BEATS ENTROPY. Report this; it undercuts the method.")
    else:
        print("  -> unresolved at this n. Report as such; do NOT claim entropy wins.")

    # A near-constant self-report cannot rank anything, which is the usual
    # failure mode for verbalized confidence and worth naming explicitly.
    if usable["mean_confidence"].nunique() <= 3:
        print(f"\n  NOTE: only {usable['mean_confidence'].nunique()} distinct "
              "confidence values -- the model is near-constant, so its AUROC is "
              "capped by ties regardless of calibration.")

n = 300   confidence parsed on 93.8% of samples, 295/300 items usable
stated confidence: min 68  median 95  max 100  distinct values 37

  perception entropy      AUROC 0.807 [0.757, 0.854]
  -verbalized confidence  AUROC 0.528 [0.463, 0.593]

  PAIRED entropy - confidence: +0.279 [+0.194, +0.359]  resolved=True
  -> entropy beats simply asking. The K forward passes are justified.


In [9]:
RUN_NAME = "confidence_perception"

# Save. Drive first, then repo + push -- Drive is the source of truth because
# pushes from Colab 403 routinely here (see notebook 15).
#
# The filename is derived from MODEL_ID, not hardcoded, so the commented 7B
# swap in the model cell cannot silently produce a CSV labelled as the 3B run.
import os
import subprocess
from datetime import datetime, timezone

timestamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
tag = "gate" if PROCESS_N < 300 else "full"
model_slug = MODEL_ID.split("/")[-1].lower()
csv_name = (f"{RUN_NAME}_{tag}_n{PROCESS_N}_"
            f"{'4bit_' if QUANTIZED else ''}{model_slug}_{timestamp}.csv")

drive_results = f"{PROJECT_DIR}/results"
os.makedirs(drive_results, exist_ok=True)
scored_df.to_csv(f"{drive_results}/{csv_name}", index=False)
print(f"Backup written to {drive_results}/{csv_name}")

os.makedirs("repo/results", exist_ok=True)
scored_df.to_csv(f"repo/results/{csv_name}", index=False)
print(f"Wrote repo/results/{csv_name} ({len(scored_df)} rows)")

_REDACT = []


def git(*args):
    r = subprocess.run(["git", "-C", "repo", *args], capture_output=True, text=True)
    out = (r.stdout or "") + (r.stderr or "")
    for s in _REDACT:
        if s:
            out = out.replace(s, "***")
    if r.returncode != 0 and out.strip():
        print(out.strip())
    return r


git("config", "user.email", "colab-pilot@localhost")
git("config", "user.name", "Colab Pilot Run")
git("add", f"results/{csv_name}")
if git("commit", "-m", f"Add {RUN_NAME} results: {csv_name}").returncode != 0:
    print("git commit failed -- CSV is safe on Drive.")

tok = (globals().get("GH_TOKEN") or "").strip()
_REDACT.append(tok)
pushed = False
if tok:
    url = REPO_URL.replace("https://", f"https://{tok}@")
    if git("fetch", url, "main").returncode == 0:
        if git("rebase", "FETCH_HEAD").returncode != 0:
            git("rebase", "--abort")
    pushed = git("push", url, "HEAD:main").returncode == 0
print("Pushed." if pushed else
      f"Push failed or skipped -- the CSV is on Drive at {drive_results}/{csv_name}, "
      "nothing is lost.")


Backup written to /content/drive/MyDrive/uncertainty-math-vlm/results/confidence_perception_full_n300_qwen2.5-vl-3b-instruct_20260811T000156Z.csv
Wrote repo/results/confidence_perception_full_n300_qwen2.5-vl-3b-instruct_20260811T000156Z.csv (300 rows)
remote: Permission to sepehrmaleki369/uncertainty-math-vlm.git denied to sepehrmaleki369.
fatal: unable to access 'https://github.com/sepehrmaleki369/uncertainty-math-vlm.git/': The requested URL returned error: 403
Push failed or skipped -- the CSV is on Drive at /content/drive/MyDrive/uncertainty-math-vlm/results/confidence_perception_full_n300_qwen2.5-vl-3b-instruct_20260811T000156Z.csv, nothing is lost.


## What to carry out

- **Report the paired difference**, with its CI, as the headline. Two marginal
  AUROCs side by side invite the difference-in-significance fallacy.
- **If confidence is near-constant**, say so: the grading-arm version landed
  at a median of 82.5 with a range of 56–95, and a self-report that barely
  varies cannot rank anything whatever its calibration.
- **If asking wins, report it.** That is the outcome that would matter most,
  and the reason this notebook exists rather than being assumed away.
- CSV is Drive-only as always. Snapshot into `reference/` only if it becomes
  claim-bearing.